# Synapse: Semantic Search Diagnostic Suite

Welcome to the **Semantic Search Diagnostic Suite**.

In [ ]:
import getpass, os, json
import numpy as np
from typing import List, Dict, Any
from openai import OpenAI
client = OpenAI()
print("✅ Ready")

In [ ]:
corpus = [
    {"id": "doc_1", "text": "General: password reset instructions.", "plan": "all"},
    {"id": "doc_2", "text": "Enterprise: Contact your Account Manager for 2FA.", "plan": "enterprise"},
    {"id": "doc_3", "text": "Pro: Use the dashboard security panel for 2FA.", "plan": "pro"}
]
def get_emb(t): return client.embeddings.create(input=[t], model="text-embedding-3-small").data[0].embedding

In [ ]:
def evaluate(retrieved, expected, k=3):
    hit = 1 if expected in retrieved[:k] else 0
    try: rank = retrieved.index(expected) + 1; rr = 1/rank if rank<=k else 0
    except: rr = 0
    return {"hit": hit, "mrr": round(rr, 2)}

In [ ]:
def search(q, plan_filter=None):
    qe = np.array(get_emb(q))
    results = []
    for doc in corpus:
        if plan_filter and doc["plan"] != plan_filter and doc["plan"] != "all": continue
        de = np.array(get_emb(doc["text"]))
        results.append((doc["id"], np.dot(qe, de)))
    return [r[0] for r in sorted(results, key=lambda x: x[1], reverse=True)]

query = "how to disable 2fa"
print(f"Naive: {evaluate(search(query), 'doc_3')}")
print(f"Fixed: {evaluate(search(query, plan_filter='pro'), 'doc_3')}")